In [116]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

### Dicíonário das colunas

- identificação da vítima (número sequencial)
- pressão sistólica (pSist): [5, 22] - não usar, é utilizada no cálculo de si3
- pressão diastólica (pDiast): [0, 15] - não usar, é utilizada no cálculo de si3
- qualidade da pressão (qPA): [-10,10] onde 0 é a qualidade máxima -10 é a pior qualidade
quando a pressão está excessivamente baixa, +10 é a pior qualidade quando a pressão está
excessivamente alta
- pulso (pulso): [0,200] bpM
- respiração (resp): [0,22] FpM (frequência de respiração)
- gravidade (gravid): valor obtido pela fórmula perdida, é usado na classificação de saída
rótulo que representa a classe de saída: deve ser inferida pelo modelo

In [117]:
df = pd.read_csv('Sinais_Vitais.csv')
df.head()

,id,pSist,pDiast,qPA,pulso,resp,gravid,classe
0,1,13.592433,12.220855,8.416754,75.921057,21.635259,40.000000,2
1,2,15.775386,13.586879,8.725890,63.813564,19.718734,41.530427,2
2,3,3.649369,1.904802,0.000000,197.210213,19.045471,52.730745,3
3,4,17.264362,13.700638,8.733333,143.636181,17.621141,34.679911,2
4,5,12.705183,9.485389,1.747626,82.636672,12.209535,69.375882,3


### Random Forest

In [118]:
df.describe()

,id,pSist,pDiast,qPA,pulso,resp,gravid,classe
count,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000,1500.000000
mean,750.500000,14.888303,7.500131,2.414994,101.196190,10.854599,41.427388,2.153333
std,433.157015,4.790653,4.248160,5.128025,57.872687,6.277191,16.590523,0.694366
min,1.000000,0.435454,0.007836,-8.732919,0.046213,0.007174,13.222719,1.000000
25%,375.750000,11.944189,3.819947,-0.861911,50.511863,5.548182,27.873461,2.000000
50%,750.500000,15.550940,7.586953,4.237224,100.887445,10.963869,40.000000,2.000000
75%,1125.250000,18.715230,11.067687,6.764517,151.354666,16.412472,51.958481,3.000000
max,1500.000000,21.997531,14.992374,8.733333,199.954625,21.959598,87.000000,4.000000


In [119]:
# pDist e pDiast são utilizadas para calcular qPA, então não serão usadas como features

X = df[['qPA', 'pulso', 'resp']] # features
y = df['classe'] # target

In [120]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [121]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [122]:
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.2%}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("Matriz de confusão:\n", confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 90.00%

Classification Report:
               precision    recall  f1-score   support

           1       0.94      0.87      0.90        70
           2       0.89      0.96      0.93       250
           3       0.91      0.82      0.86       121
           4       0.80      0.44      0.57         9

    accuracy                           0.90       450
   macro avg       0.88      0.77      0.82       450
weighted avg       0.90      0.90      0.90       450

Matriz de confusão:
 [[ 61   9   0   0]
 [  4 241   5   0]
 [  0  21  99   1]
 [  0   0   5   4]]


### Multi-layer Perceptron (MLP)

In [123]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

In [124]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [125]:
mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=5000, random_state=42)
mlp.fit(X_train, y_train)

y_pred_mlp = mlp.predict(X_test)

In [126]:
print(f"MLP Classifier Accuracy: {accuracy_score(y_test, y_pred_mlp):.2%}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_mlp))
print("Matriz de confusão:\n", confusion_matrix(y_test, y_pred_mlp))

MLP Classifier Accuracy: 90.44%

Classification Report:
               precision    recall  f1-score   support

           1       0.84      0.83      0.83        70
           2       0.90      0.94      0.92       250
           3       0.95      0.88      0.91       121
           4       1.00      0.89      0.94         9

    accuracy                           0.90       450
   macro avg       0.92      0.88      0.90       450
weighted avg       0.91      0.90      0.90       450

Matriz de confusão:
 [[ 58  12   0   0]
 [ 11 235   4   0]
 [  0  15 106   0]
 [  0   0   1   8]]


### Teste Cego

O dataset para o teste segue quase o mesmo formato dos dados históricos. No entanto, retiramos si1, si2, g1 e y1. Este arquivo vai ser utilizado somente na fase de teste cego do modelo aprendido para cada um dos classificadores (Random Forest e Rede Neural MLP).

In [127]:
blind_df = pd.read_csv('teste_cego_com_classe.csv')

In [128]:
# Prepare blind test set
blind_features = ['qPA','pulso','resp']
X_blind = blind_df[blind_features]
X_blind_mlp = scaler.transform(X_blind)

In [129]:
# Predict using both models
pred_rf = rf.predict(X_blind)
pred_mlp = mlp.predict(X_blind_mlp)

In [130]:
# Adicionar as previsões ao DataFrame cego
blind_df['pred_classe_rf'] = pred_rf
blind_df['pred_classe_mlp'] = pred_mlp
blind_df.head()

,id,qPA,pulso,resp,classe,pred_classe_rf,pred_classe_mlp
0,1,8.665540,24.786456,6.983413,1,1,1
1,2,8.733333,98.071459,17.919875,3,3,3
2,3,0.000000,170.599631,17.359617,3,3,3
3,4,8.733333,174.785481,18.749806,2,2,2
4,5,-0.000000,54.058866,11.124006,3,3,3


In [131]:
# Accuracy on blind test set
print(f"Blind Test Random Forest Accuracy: {accuracy_score(blind_df['classe'], blind_df['pred_classe_rf']):.2%}")
print(f"Blind Test MLP Classifier Accuracy: {accuracy_score(blind_df['classe'], blind_df['pred_classe_mlp']):.2%}")


Blind Test Random Forest Accuracy: 93.64%
Blind Test MLP Classifier Accuracy: 92.12%


In [132]:
# Print only where column classe is not equal to pred_classe_rf or pred_classe_mlp
blind_df[(blind_df['classe'] != blind_df['pred_classe_rf']) | (blind_df['classe'] != blind_df['pred_classe_mlp'])]


,id,qPA,pulso,resp,classe,pred_classe_rf,pred_classe_mlp
6,7,-3.365413,112.995914,20.919677,3,3,2
21,22,-4.336624,12.680900,15.967056,3,2,2
28,29,4.748462,35.520492,5.837085,2,1,2
34,35,-8.726313,59.106767,3.973572,2,2,1
42,43,5.560232,69.575839,20.032540,2,3,3
52,53,4.747456,141.383484,9.936053,1,1,2
73,74,0.001500,88.227119,22.035443,2,4,4
76,77,-4.359020,119.442270,20.672542,2,3,2
96,97,4.137194,59.648794,11.260038,3,2,2
97,98,4.622221,9.270253,1.817040,2,2,1
